# NSCLC Regimen Registry Review

This notebook loads the NSCLC regimen registry through the profile manifest and renders it in a review-friendly format.


In [1]:
from pathlib import Path
import os
import sys


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "scoop").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not locate the SCOOP repo root.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CONFIG_ROOT = ROOT / "scoop" / "configuration"
from collections import Counter
from html import escape
import sqlalchemy as sa
import sqlalchemy.orm as so
import yaml
from IPython.display import HTML, Markdown, display

from scoop.configuration.config_loader import load_profile_manifest
from scoop.configuration.regimen_registry import load_regimen_registry
from scoop.utils.hemonc_query import build_variant_query_artifacts, execute_variant_query


2026-04-06 22:47:09,827 | DEBUG   | scoop | SCOOP logging initialized


In [2]:
manifest = load_profile_manifest('nsclc', base_dir=CONFIG_ROOT)
REGISTRY_PATH = CONFIG_ROOT / 'profiles' / 'nsclc' / manifest.regimen_registry
REGISTRY_MODEL = load_regimen_registry(REGISTRY_PATH)

with REGISTRY_PATH.open('r', encoding='utf-8') as handle:
    REGISTRY = yaml.safe_load(handle)['registry']

ENTRIES = REGISTRY['entries']
HEMONC_GROUPS = REGISTRY.get('hemonc_groups', {})
manifest


ProfileManifest(profile_id='nsclc', default_graph='nsclc', graphs={'nsclc': PosixPath('nsclc.yaml'), 'nsclc_stg1': PosixPath('nsclc_stg1.yaml'), 'nsclc_stg23_resectable': PosixPath('nsclc_stg23_resectable.yaml'), 'nsclc_stg3_unresectable': PosixPath('nsclc_stg3_unresectable.yaml'), 'nsclc_stg4_driver': PosixPath('nsclc_stg4_driver.yaml'), 'nsclc_stg4_nodrivers': PosixPath('nsclc_stg4_nodrivers.yaml')}, regimen_registry=PosixPath('nsclc_regimen_registry.yaml'), concept_map=PosixPath('../../mappers/concept_maps.yaml'), measurement_map=PosixPath('../../mappers/measurement_mappers.yaml'), observation_map=PosixPath('../../mappers/observation_mappers.yaml'), entry_graph='nsclc', pathways={'nsclc_stage1': 'nsclc_stg1', 'nsclc_stage23_resectable': 'nsclc_stg23_resectable', 'nsclc_stage3_unresectable': 'nsclc_stg3_unresectable', 'nsclc_stage4_driver': 'nsclc_stg4_driver', 'nsclc_stage4_nodrivers': 'nsclc_stg4_nodrivers'})

In [3]:
public_entries = {k: v for k, v in ENTRIES.items() if v['public']}
internal_entries = {k: v for k, v in ENTRIES.items() if not v['public']}

{
    'public_entries': len(public_entries),
    'internal_entries': len(internal_entries),
    'hemonc_groups': len(HEMONC_GROUPS),
    'clinical_groups': Counter(entry['clinical_group'] for entry in ENTRIES.values()),
}


{'public_entries': 36,
 'internal_entries': 8,
 'hemonc_groups': 26,
 'clinical_groups': Counter({'metastatic_non_driver': 15,
          'metastatic_driver_positive': 11,
          'unresectable_stage_iii': 7,
          'resectable_stage_ii_iii': 6,
          'local_stage_i': 5})}

In [4]:
GROUP_ORDER = [
    'local_stage_i',
    'resectable_stage_ii_iii',
    'unresectable_stage_iii',
    'metastatic_driver_positive',
    'metastatic_non_driver',
]


def render_entry(entry_id: str, entry: dict) -> str:
    aliases = ', '.join(entry.get('aliases', [])) or 'None'
    graph_sources = ', '.join(entry['graph_sources']) or 'None'
    exploratory_sources = ', '.join(entry.get('exploratory_sources', [])) or 'None'
    required_modifiers = ', '.join(entry.get('required_modifiers', [])) or 'None'
    notes = escape(entry.get('notes') or 'None')
    resolver = entry['resolver']

    resolver_lines = [f"<li><b>type</b>: {escape(resolver['type'])}</li>"]
    if resolver['type'] == 'component_query':
        resolved = REGISTRY_MODEL.resolve_hemonc_query(entry_id)
        group_id = resolver.get('group_id')
        if group_id:
            group = HEMONC_GROUPS[group_id]
            resolver_lines.append(
                f"<li><b>group_id</b>: <code>{escape(group_id)}</code></li>"
            )
            resolver_lines.append(
                f"<li><b>basis</b>: {escape(group.get('basis', 'variant'))}</li>"
            )
            resolver_lines.append(
                f"<li><b>condition_set</b>: {escape(str(group.get('condition_set', '')) or 'None')}</li>"
            )
        if resolved is not None:
            resolver_lines.append(
                f"<li><b>resolved_condition_cuis</b>: {escape(str(resolved.default_condition_cuis))}</li>"
            )
            resolver_lines.append(
                f"<li><b>component_lists</b>: {escape(str(resolved.component_lists))}</li>"
            )
            resolver_lines.append(
                f"<li><b>category_restrictions</b>: {escape(str(resolved.category_restrictions))}</li>"
            )
        else:
            resolver_lines.append(
                f"<li><b>default_condition_cuis</b>: {resolver.get('default_condition_cuis', [])}</li>"
            )
            resolver_lines.append(
                f"<li><b>component_lists</b>: {escape(str(resolver.get('component_lists', [])))}</li>"
            )
            resolver_lines.append(
                f"<li><b>category_restrictions</b>: {escape(str(resolver.get('category_restrictions', {})))}</li>"
            )
    elif resolver['type'] == 'composite':
        resolver_lines.append(
            f"<li><b>members</b>: {escape(', '.join(resolver.get('members', [])))}</li>"
        )

    header = f"<summary><code>{escape(entry_id)}</code> ({'public' if entry['public'] else 'internal'})</summary>"
    body_lines = [
        f"<div><b>clinical_group</b>: {escape(entry['clinical_group'])}</div>",
        f"<div><b>stage_scope</b>: {escape(', '.join(entry['stage_scope']))}</div>",
        f"<div><b>aliases</b>: {escape(aliases)}</div>",
        f"<div><b>graph_sources</b>: {escape(graph_sources)}</div>",
        f"<div><b>exploratory_sources</b>: {escape(exploratory_sources)}</div>",
        f"<div><b>required_modifiers</b>: {escape(required_modifiers)}</div>",
        f"<div><b>notes</b>: {notes}</div>",
        "<div><b>resolver</b>:</div>",
        f"<ul>{''.join(resolver_lines)}</ul>",
    ]
    body = ''.join(body_lines)
    return (
        "<details style='margin: 0.4rem 0;'>"
        f"{header}<div style='margin: 0.5rem 0 0 1rem;'>{body}</div></details>"
    )


def render_group(group_name: str, public_only: bool) -> str:
    subset = {
        entry_id: entry
        for entry_id, entry in ENTRIES.items()
        if entry['clinical_group'] == group_name and entry['public'] == public_only
    }
    if not subset:
        return ''
    title = group_name.replace('_', ' ')
    entries_html = ''.join(render_entry(entry_id, entry) for entry_id, entry in subset.items())
    return f"<h3>{escape(('Public' if public_only else 'Internal') + ': ' + title)}</h3>{entries_html}"


In [5]:
public_html = ''.join(render_group(group_name, public_only=True) for group_name in GROUP_ORDER)
display(HTML(public_html))


In [6]:
internal_html = ''.join(render_group(group_name, public_only=False) for group_name in GROUP_ORDER)
display(HTML(internal_html))


In [7]:
def show_entry(entry_id: str):
    display(HTML(render_entry(entry_id, ENTRIES[entry_id])))


def show_raw(entry_id: str):
    return ENTRIES[entry_id]


def show_resolved_query(entry_id: str):
    resolved = REGISTRY_MODEL.resolve_hemonc_query(entry_id)
    if resolved is None:
        display(Markdown(f"`{entry_id}` does not resolve to a direct HemOnc component query."))
        return None
    return resolved.model_dump()


def compile_hemonc_query(entry_id: str):
    resolved = REGISTRY_MODEL.resolve_hemonc_query(entry_id)
    if resolved is None:
        raise ValueError(f"{entry_id} does not resolve to a direct HemOnc query.")
    try:
        artifacts = build_variant_query_artifacts(resolved)
    except ImportError as exc:
        return {
            'resolved_query': resolved.model_dump(),
            'import_error': str(exc),
        }
    return {
        'resolved_query': resolved.model_dump(),
        'variant_cui_sql': str(
            artifacts.variant_cuis_subquery.select().compile(
                compile_kwargs={'literal_binds': True}
            )
        ),
        'variants_sql': str(
            artifacts.variants_stmt.compile(compile_kwargs={'literal_binds': True})
        ),
    }


HEMONC_ENGINE_URL = os.getenv('HEMONC_ENGINE_URL')


def run_hemonc_query(entry_id: str, engine_url: str | None = None, limit: int = 20):
    resolved = REGISTRY_MODEL.resolve_hemonc_query(entry_id)
    if resolved is None:
        raise ValueError(f"{entry_id} does not resolve to a direct HemOnc query.")

    engine_url = engine_url or HEMONC_ENGINE_URL
    if not engine_url:
        raise ValueError('Set HEMONC_ENGINE_URL or pass engine_url to execute HemOnc queries.')

    engine = sa.create_engine(engine_url)
    with so.Session(engine) as session:
        variants = execute_variant_query(session, resolved)

    rows = []
    for variant in variants[:limit]:
        rows.append(
            {
                'variant_cui': getattr(variant, 'variant_cui', None),
                'variant': getattr(variant, 'variant', None),
                'regimen': getattr(variant, 'regimen', None),
                'version': getattr(variant, 'version', None),
            }
        )
    return rows


In [8]:
show_resolved_query('egfr_intervention')


{'group_id': 'egfr_intervention',
 'basis': 'variant',
 'default_condition_cuis': [30824,
  30898,
  154361,
  30910,
  52958,
  54044,
  46104,
  13094,
  642,
  25309,
  25316,
  149626,
  157580,
  18383,
  30982],
 'component_lists': [['osimertinib',
   'gefitinib',
   'erlotinib',
   'afatinib',
   'dacomitinib']],
 'category_restrictions': {'immunotherapy': {'type': 'min', 'amount': 0},
  'chemotherapy': {'type': 'min', 'amount': 1}}}

In [9]:
compile_hemonc_query('egfr_intervention')

{'resolved_query': {'group_id': 'egfr_intervention',
  'basis': 'variant',
  'default_condition_cuis': [30824,
   30898,
   154361,
   30910,
   52958,
   54044,
   46104,
   13094,
   642,
   25309,
   25316,
   149626,
   157580,
   18383,
   30982],
  'component_lists': [['osimertinib',
    'gefitinib',
    'erlotinib',
    'afatinib',
    'dacomitinib']],
  'category_restrictions': {'immunotherapy': {'type': 'min', 'amount': 0},
   'chemotherapy': {'type': 'min', 'amount': 1}}},
 'variant_cui_sql': "SELECT anon_1.variant_cui \nFROM (SELECT anon_2.variant_cui AS variant_cui \nFROM (SELECT DISTINCT anon_3.variant_cui AS variant_cui, anon_3.canmed_major_class AS canmed_major_class, anon_3.canmed_minor_class AS canmed_minor_class, anon_3.main_class AS main_class, anon_3.component AS component, anon_3.component_cui AS component_cui, coalesce(CASE WHEN (anon_3.canmed_major_class = 'Serine-threonine kinase inhibitor') THEN 'chemotherapy' WHEN (anon_3.canmed_major_class = 'MEK inhibitor') 

In [10]:
# Requires an installed `hemonc_import` package plus a reachable HemOnc DB.
# Example:
# run_hemonc_query('egfr_intervention', engine_url='postgresql+psycopg://...')
